In [1]:
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
from requests.adapters import HTTPAdapter
from sqlalchemy import create_engine, text
from urllib3.util.retry import Retry

load_dotenv(find_dotenv(usecwd=True))
engine = create_engine(os.environ["DATABASE_URL"])

PROJECT_DIR = Path(find_dotenv(usecwd=True)).parent
RAW_DIR = PROJECT_DIR / "data" / "raw" / "amfi"
RAW_DIR.mkdir(parents=True, exist_ok=True)

AMFI_URL = "https://portal.amfiindia.com/spages/NAVAll.txt"
print("Raw files will be saved in:", RAW_DIR)

Raw files will be saved in: D:\FundLens\data\raw\amfi


In [2]:
DDL = """
CREATE SCHEMA IF NOT EXISTS raw;

CREATE TABLE IF NOT EXISTS raw.amfi_nav_daily (
    scheme_code      INTEGER       NOT NULL,
    isin_growth      TEXT,
    isin_reinvest    TEXT,
    scheme_name      TEXT          NOT NULL,
    plan             TEXT,
    option           TEXT,
    nav              NUMERIC(18,4) NOT NULL,
    nav_date         DATE          NOT NULL,
    scheme_type      TEXT,
    scheme_category  TEXT,
    fund_house       TEXT,
    source_file      TEXT,
    loaded_at        TIMESTAMPTZ   NOT NULL DEFAULT now(),
    PRIMARY KEY (scheme_code, nav_date)
);

CREATE TABLE IF NOT EXISTS raw.load_log (
    run_id         SERIAL PRIMARY KEY,
    source         TEXT        NOT NULL,
    started_at     TIMESTAMPTZ NOT NULL,
    finished_at    TIMESTAMPTZ,
    rows_read      INTEGER,
    rows_loaded    INTEGER,
    rows_rejected  INTEGER,
    status         TEXT,
    message        TEXT
);
"""
with engine.begin() as conn:
    conn.execute(text(DDL))
print("raw schema ready")

raw schema ready


In [3]:
def http_session():
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers["User-Agent"] = "FundLens/1.0 (portfolio data project)"
    return session


def download_amfi():
    resp = http_session().get(AMFI_URL, timeout=60)
    resp.raise_for_status()
    path = RAW_DIR / f"NAVAll_{datetime.now():%Y%m%d_%H%M}.txt"
    path.write_text(resp.text, encoding="utf-8")
    return path


started_at = datetime.now()
raw_file = download_amfi()
print(f"Saved {raw_file.name}: {raw_file.stat().st_size / 1024:,.0f} KB")

Saved NAVAll_20261013_1013.txt: 1,503 KB


In [4]:
content = raw_file.read_text(encoding="utf-8")
lines = content.splitlines()
print("Total lines:", len(lines))
print()
for line in lines[:12]:
    print(repr(line))

Total lines: 36224

'Scheme Code;ISIN Div Payout/ ISIN Growth;ISIN Div Reinvestment;Scheme Name;Plan;Option;Net Asset Value;Date'
''
' '
''
"Open Ended Schemes(Childrenâ\x80\x99s Fund - Childrens' Fund)"
''
' '
''
'Axis Mutual Fund'
''
' '
''


In [5]:
CATEGORY_RE = re.compile(r"^(?P<type>[A-Za-z ]+Schemes?)\s*\((?P<category>.+)\)\s*$")
COLUMN_NAMES = {
    "scheme_code": "Scheme Code",
    "isin_growth": "ISIN Div Payout/ ISIN Growth",
    "isin_reinvest": "ISIN Div Reinvestment",
    "scheme_name": "Scheme Name",
    "plan": "Plan",
    "option": "Option",
    "nav": "Net Asset Value",
    "nav_date": "Date",
}


def parse_amfi(content, source_file):
    lines = content.splitlines()
    header = [h.strip() for h in lines[0].split(";")]
    idx = {key: header.index(name) for key, name in COLUMN_NAMES.items() if name in header}
    missing = {"scheme_code", "scheme_name", "nav", "nav_date"} - idx.keys()
    if missing:
        raise ValueError(f"AMFI file format changed, missing columns: {missing}")

    rows, rejects = [], []
    scheme_type = scheme_category = fund_house = None

    for line_no, raw in enumerate(lines[1:], start=2):
        line = raw.strip()
        if not line:
            continue                                   # skip empty lines

        if ";" not in line:                            # a header line, not data
            m = CATEGORY_RE.match(line)
            if m:                                      # category header
                scheme_type = m["type"].strip()
                scheme_category = m["category"].strip()
                fund_house = None
            else:                                      # fund house header
                fund_house = line
            continue

        parts = [p.strip() for p in line.split(";")]
        get = lambda key: parts[idx[key]] if key in idx and idx[key] < len(parts) else None
        record = {key: get(key) for key in COLUMN_NAMES}
        record.update(scheme_type=scheme_type, scheme_category=scheme_category,
                      fund_house=fund_house, source_file=source_file)

        # ---- validation: every problem gets a reason ----
        reason = None
        try:
            record["scheme_code"] = int(record["scheme_code"])
        except (TypeError, ValueError):
            reason = "bad scheme_code"
        try:
            record["nav"] = float(record["nav"])
            if record["nav"] <= 0:
                reason = reason or "nav not positive"
        except (TypeError, ValueError):
            reason = reason or f"nav not a number ({record['nav']})"
        try:
            record["nav_date"] = datetime.strptime(record["nav_date"], "%d-%b-%Y").date()
        except (TypeError, ValueError):
            reason = reason or f"bad date ({record['nav_date']})"
        for key in ("isin_growth", "isin_reinvest"):
            if record[key] in ("-", ""):
                record[key] = None

        if reason:
            rejects.append({"line_no": line_no, "reason": reason, "line": line})
        else:
            rows.append(record)

    df = pd.DataFrame(rows)
    before = len(df)
    df = df.drop_duplicates(subset=["scheme_code", "nav_date"], keep="last")
    if len(df) < before:
        rejects.append({"line_no": None, "reason": f"{before - len(df)} duplicate rows dropped", "line": ""})
    return df, pd.DataFrame(rejects)

In [6]:
nav_df, rejects_df = parse_amfi(content, raw_file.name)
print(f"Valid rows: {len(nav_df):,}")
print(f"Rejected:   {len(rejects_df):,}")
nav_df.head(10)

Valid rows: 14,136
Rejected:   241


,scheme_code,isin_growth,isin_reinvest,scheme_name,plan,option,nav,nav_date,scheme_type,scheme_category,fund_house,source_file
0,135762,INF846K01WO1,NaN,Axis Children's Fund,Direct Plan,Growth Option,28.9937,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
1,135765,INF846K01WP8,NaN,Axis Children's Fund,Direct Plan,IDCW Option,26.7085,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
2,135759,INF846K01WJ1,NaN,Axis Children's Fund,Regular Plan,Growth Option,25.2325,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
3,135760,INF846K01WK9,NaN,Axis Children's Fund,Regular Plan,IDCW Option,23.2790,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
4,135764,INF846K01WR4,NaN,Axis Children's Fund,Direct Plan,Growth Option,29.5482,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
5,135763,INF846K01WS2,INF846K01WQ6,Axis Children's Fund,Direct Plan,IDCW Option,26.7589,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
6,135766,INF846K01WM5,NaN,Axis Children's Fund,Regular Plan,Growth Option,25.2294,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
7,135761,INF846K01WN3,INF846K01WL7,Axis Children's Fund,Regular Plan,IDCW Option,23.2580,2026-10-12,Open Ended Schemes,Childrenâs Fund - Childrens' Fund,Axis Mutual Fund,NAVAll_20261013_1013.txt
8,129008,INF090I01KR8,NaN,Franklin India Banking & PSU Debt Fund,Direct Plan,Growth,25.3336,2026-10-12,Open Ended Schemes,Debt Scheme - Banking and PSU Fund,Franklin Templeton Mutual Fund,NAVAll_20261013_1013.txt
9,129006,INF090I01KO5,NaN,Franklin India Banking & PSU Debt Fund,Regular Plan,Growth,24.1582,2026-10-12,Open Ended Schemes,Debt Scheme - Banking and PSU Fund,Franklin Templeton Mutual Fund,NAVAll_20261013_1013.txt


In [7]:
rejects_df.head(10)

,line_no,reason,line
0,537,nav not positive,148304;INF090I01VZ8;-;Franklin India Credit Ri...
1,539,nav not positive,148303;INF090I01VX3;-;Franklin India Credit Ri...
2,541,nav not positive,147956;INF090I01TK4;-;Franklin India Credit Ri...
3,543,nav not positive,147955;INF090I01TL2;-;Franklin India Credit Ri...
4,545,nav not positive,147957;INF090I01TI8;-;Franklin India Credit Ri...
5,547,nav not positive,147954;INF090I01TJ6;-;Franklin India Credit Ri...
6,549,nav not positive,147959;INF090I01TM0;-;Franklin India Credit Ri...
7,551,nav not positive,147958;INF090I01TP3;-;Franklin India Credit Ri...
8,553,nav not positive,147961;INF090I01TN8;-;Franklin India Credit Ri...
9,555,nav not positive,148305;INF090I01VW5;-;Franklin India Credit Ri...


In [8]:
nav_df["fund_house"].value_counts().head(10)

fund_house
ICICI Prudential Mutual Fund         2501
Nippon India Mutual Fund             1820
UTI Mutual Fund                      1304
Kotak Mahindra Mutual Fund            939
HDFC Mutual Fund                      575
SBI Mutual Fund                       493
Bandhan Mutual Fund                   434
Aditya Birla Sun Life Mutual Fund     410
Axis Mutual Fund                      409
Tata Mutual Fund                      401
Name: count, dtype: int64

In [9]:
nav_df["plan"].value_counts()

plan
                5586
Regular Plan    4289
Direct Plan     4261
Name: count, dtype: int64

In [10]:
LOAD_COLUMNS = ["scheme_code", "isin_growth", "isin_reinvest", "scheme_name", "plan", "option",
                "nav", "nav_date", "scheme_type", "scheme_category", "fund_house", "source_file"]

UPSERT = f"""
INSERT INTO raw.amfi_nav_daily ({", ".join(LOAD_COLUMNS)})
SELECT {", ".join(LOAD_COLUMNS)} FROM tmp_amfi
ON CONFLICT (scheme_code, nav_date) DO UPDATE SET
    nav = EXCLUDED.nav, scheme_name = EXCLUDED.scheme_name, plan = EXCLUDED.plan,
    option = EXCLUDED.option, scheme_type = EXCLUDED.scheme_type,
    scheme_category = EXCLUDED.scheme_category, fund_house = EXCLUDED.fund_house,
    isin_growth = EXCLUDED.isin_growth, isin_reinvest = EXCLUDED.isin_reinvest,
    source_file = EXCLUDED.source_file, loaded_at = now()
"""


def load_nav(df):
    with engine.begin() as conn:                   # one transaction: all or nothing
        df[LOAD_COLUMNS].to_sql("tmp_amfi", conn, if_exists="replace", index=False)
        loaded = conn.execute(text(UPSERT)).rowcount
        conn.execute(text("DROP TABLE tmp_amfi"))
    return loaded


def log_run(status, rows_read, loaded, rejected, message=""):
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO raw.load_log (source, started_at, finished_at, rows_read, rows_loaded,
                                      rows_rejected, status, message)
            VALUES ('amfi_daily', :s, now(), :r, :l, :j, :st, :m)"""),
            {"s": started_at, "r": rows_read, "l": loaded, "j": rejected, "st": status, "m": message})


try:
    loaded = load_nav(nav_df)
    log_run("success", len(nav_df) + len(rejects_df), loaded, len(rejects_df), raw_file.name)
    print(f"Upserted {loaded:,} rows into raw.amfi_nav_daily")
except Exception as e:
    log_run("failed", len(nav_df), 0, len(rejects_df), str(e)[:500])
    raise

Upserted 14,136 rows into raw.amfi_nav_daily


In [11]:
pd.read_sql("""
    SELECT nav_date, COUNT(*) AS schemes, COUNT(DISTINCT fund_house) AS fund_houses
    FROM raw.amfi_nav_daily
    GROUP BY nav_date ORDER BY nav_date DESC LIMIT 5
""", engine)

,nav_date,schemes,fund_houses
0,2026-10-12,8541,54
1,2026-10-09,92,11
2,2026-10-08,8,1
3,2026-10-06,4,1
4,2026-10-05,5,2


In [12]:
pd.read_sql("SELECT * FROM raw.load_log ORDER BY run_id DESC", engine)

,run_id,source,started_at,finished_at,rows_read,rows_loaded,rows_rejected,status,message
0,1,amfi_daily,2026-10-13 04:43:47.013651+00:00,2026-10-13 04:45:59.462033+00:00,14377,14136,241,success,NAVAll_20261013_1013.txt
